# Key enrollment and rotation

Open this `.ipynb` in JupyterLab or an editor with Jupyter support. Install Deno's kernel with
`deno jupyter --install`, select **Deno**, and Run All from a fresh kernel. This notebook imports
exact published verifier **2.0.0**. It generates its own demonstration keys, asserts success and
refusal, prints only safe semantic outcomes and cleans up resources. The saved text is the expected
semantic output; CI executes each cell in the real kernel and compares it. Random key and signature
bytes are never persisted. See [runner instructions](README.md).

Verification is not authority. The host owns trusted inputs, replay persistence, revocation and
policy. The HTTP demonstration's memory reservations last one process; use the durable-store
recipe for a persistent host.


In [ ]:
import assertPackage from "node:assert/strict";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
import { BACKEND_ID as notebookBackend } from "npm:@bounded-authority-protocol/verifier@2.0.0";
assertPackage.equal(verifierPackage.version, "2.0.0");
assertPackage.equal(notebookBackend, "portable");
console.log("verifier 2.0.0; portable backend");


verifier 2.0.0; portable backend


In [ ]:
import assert from "node:assert/strict";
import { generateKeyPairSync, sign, type KeyObject } from "node:crypto";
import {
  acceptablePublicKey, torsionFree, assembleCompact, grantSigningInput, jwkFromPublicKey, thumbprint, untrustedKeyLocator, verifyGrant,
  type GrantFacts, type Result, type SigningInput, type TrustedIssuer,
} from "npm:@bounded-authority-protocol/verifier@2.0.0";

const raw = (key: KeyObject): Uint8Array => Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const message = (i: SigningInput): Uint8Array => Buffer.concat([i.protectedSegment, Buffer.from("."), i.payloadSegment]);
const holder = raw(generateKeyPairSync("ed25519").publicKey);

function issue(keyId: string, privateKey: KeyObject, grantId: string): Uint8Array {
  const input = grantSigningInput({
    keyId, issuer: "https://issuer.example", grantId, audiences: ["https://storage.example"],
    issuedAt: 1_000, notBefore: 1_000, expiresAt: 100_000,
    holderThumbprint: thumbprint(jwkFromPublicKey(holder)),
    operations: [{ name: "objects.list", selectors: ["all"] }],
  });
  assert.ok(input.ok);
  const compact = assembleCompact(input.value, sign(null, message(input.value), privateKey));
  assert.ok(compact.ok);
  return compact.value;
}

const oldKey = generateKeyPairSync("ed25519");
const newKey = generateKeyPairSync("ed25519");
const trusted = new Map<string, TrustedIssuer>([
  ["issuer-2026-09", { keyId: "issuer-2026-09", publicKey: raw(oldKey.publicKey) }],
  ["issuer-2026-10", { keyId: "issuer-2026-10", publicKey: raw(newKey.publicKey) }],
]);

function verifyAny(grant: Uint8Array): Result<GrantFacts> {
  const locator = untrustedKeyLocator(grant);
  const key = locator.ok ? trusted.get(locator.value.keyId) : undefined;
  if (key === undefined) return { ok: false };
  return verifyGrant(grant, key, {
    issuer: "https://issuer.example", audience: "https://storage.example", evaluationTime: 2_000, clockSkew: 60,
  });
}

assert.equal(acceptablePublicKey(raw(newKey.publicKey)), true);
assert.equal(torsionFree(raw(newKey.publicKey)), true);
assert.equal(torsionFree(new Uint8Array(32)), false);
const before = issue("issuer-2026-09", oldKey.privateKey, "grant-a");
const after = issue("issuer-2026-10", newKey.privateKey, "grant-b");
assert.ok(verifyAny(before).ok); // both verify during the overlap
assert.ok(verifyAny(after).ok);

// A grant whose kid names the new key but which the old key signed: the signature binds it.
assert.deepEqual(verifyAny(issue("issuer-2026-10", oldKey.privateKey, "grant-c")), { ok: false });

// Once the old grants have expired, retire the old key.
trusted.delete("issuer-2026-09");
assert.deepEqual(verifyAny(before), { ok: false });
assert.ok(verifyAny(after).ok);

console.log("rotation: overlap verified; wrong signer and retired key refused");


rotation: overlap verified; wrong signer and retired key refused
